In [67]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score 

# Load data


In [68]:
# Load data
df = pd.read_csv("Energy_consumption.csv")
TARGET_COL = "EnergyConsumption"
df.head()

,Timestamp,Temperature,Humidity,SquareFootage,Occupancy,HVACUsage,LightingUsage,RenewableEnergy,DayOfWeek,Holiday,EnergyConsumption
0,2022-01-01 00:00:00,25.139433,43.431581,1565.693999,5,On,Off,2.774699,Monday,No,75.364373
1,2022-01-01 01:00:00,27.731651,54.225919,1411.064918,1,On,On,21.831384,Saturday,No,83.401855
2,2022-01-01 02:00:00,28.704277,58.907658,1755.715009,2,Off,Off,6.764672,Sunday,No,78.270888
3,2022-01-01 03:00:00,20.080469,50.371637,1452.316318,1,Off,On,8.623447,Wednesday,No,56.519850
4,2022-01-01 04:00:00,23.097359,51.401421,1094.130359,9,On,Off,3.071969,Friday,No,70.811732


In [69]:
# Drop Timestamp column (not needed for model)

print("Columns before preprocessing:", df.columns.tolist())

# Identify categorical and numerical columns
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Remove target from numerical columns
if TARGET_COL in numerical_cols:
    numerical_cols.remove(TARGET_COL)

print(f"Categorical columns: {categorical_cols}")
print(f"Numerical columns: {numerical_cols}")


Columns before preprocessing: ['Timestamp', 'Temperature', 'Humidity', 'SquareFootage', 'Occupancy', 'HVACUsage', 'LightingUsage', 'RenewableEnergy', 'DayOfWeek', 'Holiday', 'EnergyConsumption']
Categorical columns: ['Timestamp', 'HVACUsage', 'LightingUsage', 'DayOfWeek', 'Holiday']
Numerical columns: ['Temperature', 'Humidity', 'SquareFootage', 'Occupancy', 'RenewableEnergy']


# One-hot encoding

In [70]:
categorical_cols.remove("DayOfWeek")

In [71]:
print("catgerical columns after removing DayOfWeek:", categorical_cols)

catgerical columns after removing DayOfWeek: ['Timestamp', 'HVACUsage', 'LightingUsage', 'Holiday']


# Preprocess the dayofweek  feature

In [72]:
df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")
df["day_of_week"] = df["Timestamp"].dt.dayofweek
df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)

In [73]:
df.drop(columns=["Timestamp", "day_of_week", "DayOfWeek"], inplace=True)


In [74]:
df.head()

,Temperature,Humidity,SquareFootage,Occupancy,HVACUsage,LightingUsage,RenewableEnergy,Holiday,EnergyConsumption,dow_sin,dow_cos
0,25.139433,43.431581,1565.693999,5,On,Off,2.774699,No,75.364373,-0.974928,-0.222521
1,27.731651,54.225919,1411.064918,1,On,On,21.831384,No,83.401855,-0.974928,-0.222521
2,28.704277,58.907658,1755.715009,2,Off,Off,6.764672,No,78.270888,-0.974928,-0.222521
3,20.080469,50.371637,1452.316318,1,Off,On,8.623447,No,56.519850,-0.974928,-0.222521
4,23.097359,51.401421,1094.130359,9,On,Off,3.071969,No,70.811732,-0.974928,-0.222521


In [75]:
categorical_cols.remove("Timestamp")

In [76]:
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print("Columns after one-hot encoding:", df_encoded.columns.tolist())
print("Dataset shape after encoding:", df_encoded.shape)

Columns after one-hot encoding: ['Temperature', 'Humidity', 'SquareFootage', 'Occupancy', 'RenewableEnergy', 'EnergyConsumption', 'dow_sin', 'dow_cos', 'HVACUsage_On', 'LightingUsage_On', 'Holiday_Yes']
Dataset shape after encoding: (1000, 11)


# Standarization 

In [77]:
# Separate target variable
X = df_encoded.drop(columns=[TARGET_COL])
y = df_encoded[[TARGET_COL]].copy()

# Initialize scaler for features
feature_scaler = StandardScaler()
X_scaled = feature_scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

# Initialize scaler for target
target_scaler = StandardScaler()
y_scaled = target_scaler.fit_transform(y)
y_scaled = pd.Series(y_scaled.flatten(), name=TARGET_COL)

print("Features standardized (mean ≈ 0, std ≈ 1)")
print("\nFeature statistics after standardization:")
print(X_scaled.describe())
print("\nTarget statistics after standardization:")
print(y_scaled.describe())


Features standardized (mean ≈ 0, std ≈ 1)

Feature statistics after standardization:
        Temperature      Humidity  SquareFootage     Occupancy  \
count  1.000000e+03  1.000000e+03   1.000000e+03  1.000000e+03   
mean   5.773160e-16 -6.075140e-16  -4.831691e-16 -1.234568e-16   
std    1.000500e+00  1.000500e+00   1.000500e+00  1.000500e+00   
min   -1.754393e+00 -1.806234e+00  -1.732861e+00 -1.599419e+00   
25%   -8.241978e-01 -8.335862e-01  -8.774409e-01 -9.011354e-01   
50%   -8.125375e-02  6.773088e-02   2.745624e-02  1.462905e-01   
75%    8.591808e-01  8.250085e-01   8.335374e-01  8.445744e-01   
max    1.769271e+00  1.711601e+00   1.734214e+00  1.542858e+00   

       RenewableEnergy       dow_sin       dow_cos  HVACUsage_On  \
count     1.000000e+03  1.000000e+03  1.000000e+03  1.000000e+03   
mean     -8.171241e-17  4.263256e-17 -9.947598e-17  5.506706e-17   
std       1.000500e+00  1.000500e+00  1.000500e+00  1.000500e+00   
min      -1.730378e+00 -1.380244e+00 -1.287638e+

In [78]:
print("data set after preprocessing:")
print(X_scaled.head())
print(y_scaled.head())

data set after preprocessing:
   Temperature  Humidity  SquareFootage  Occupancy  RenewableEnergy   dow_sin  \
0     0.055514 -0.230642       0.227705   0.146290        -1.413722 -1.380244   
1     0.969738  1.037096      -0.308690  -1.250277         0.766292 -1.380244   
2     1.312764  1.586942       0.886871  -0.901135        -0.957284 -1.380244   
3    -1.728681  0.584431      -0.165593  -1.250277        -0.744647 -1.380244   
4    -0.664684  0.705374      -1.408109   1.542858        -1.379716 -1.380244   

    dow_cos  HVACUsage_On  LightingUsage_On  Holiday_Yes  
0 -0.325716      1.016130         -0.982159    -0.936041  
1 -0.325716      1.016130          1.018165    -0.936041  
2 -0.325716     -0.984126         -0.982159    -0.936041  
3 -0.325716     -0.984126          1.018165    -0.936041  
4 -0.325716      1.016130         -0.982159    -0.936041  
0   -0.207800
1    0.779601
2    0.149264
3   -2.522841
4   -0.767090
Name: EnergyConsumption, dtype: float64


In [79]:
#split data into train and test sets and validation set
X_train, X_temp, y_train, y_temp = train_test_split(X_scaled, y_scaled, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)
print(f"Training set shape: {X_train.shape}, {y_train.shape}")
print(f"Validation set shape: {X_val.shape}, {y_val.shape}")
print(f"Test set shape: {X_test.shape}, {y_test.shape}")

Training set shape: (700, 10), (700,)
Validation set shape: (150, 10), (150,)
Test set shape: (150, 10), (150,)


In [92]:
class EBM(nn.Module):
    def __init__(self, input_dim, hidden_dim=32, dropout_rate=0.3):
        super(EBM, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim, 1)
        )
    
    def forward(self, x):
        return self.model(x)

In [93]:
input_dim = X_train.shape[1]
model = EBM(input_dim, hidden_dim=32, dropout_rate=0.3)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=0.0001)
#negative sampling for EBM
num_epochs = 500
batch_size = 32
patience = 30
best_val_loss = float('inf')
patience_counter = 0
X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
X_val_tensor = torch.tensor(X_val.values, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val.values, dtype=torch.float32).view(-1, 1)
train_losses = []
val_losses = []

In [89]:
def sample_negative(y):
    noise = torch.randn_like(y) * 0.5
    return y + noise

In [94]:
for epoch in range(num_epochs):
    model.train()
    y_neg = sample_negative(y_train_tensor)
    permutation = torch.randperm(X_train_tensor.size()[0])
    epoch_loss = 0.0
    
    for i in range(0, X_train_tensor.size()[0], batch_size):
        indices = permutation[i:i+batch_size]
        batch_X, batch_y = X_train_tensor[indices], y_train_tensor[indices]
        
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_train_loss = epoch_loss / (X_train_tensor.size()[0] / batch_size)
    train_losses.append(avg_train_loss)
    
    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val_tensor)
        val_loss = criterion(val_outputs, y_val_tensor).item()
        val_losses.append(val_loss)
    
    # Early stopping logic
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        # Save best model
        best_model_state = model.state_dict().copy()
    else:
        patience_counter += 1
    
    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {avg_train_loss:.4f}, Val Loss: {val_loss:.4f}')
    
    # Stop early if no improvement
    if patience_counter >= patience:
        print(f'Early stopping at epoch {epoch+1} (best val loss: {best_val_loss:.4f})')
        model.load_state_dict(best_model_state)
        break

model.eval()
X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)
with torch.no_grad():
    test_outputs = model(X_test_tensor)
    test_loss = criterion(test_outputs, y_test_tensor).item()
    print(f'Test Loss: {test_loss:.4f}')
y_test_pred = target_scaler.inverse_transform(test_outputs.numpy())
y_test_true = target_scaler.inverse_transform(y_test_tensor.numpy())
mse = mean_squared_error(y_test_true, y_test_pred)
r2 = r2_score(y_test_true, y_test_pred)
print(f'Test MSE: {mse:.4f}')
print(f'Test R²: {r2:.4f}')

Epoch [10/500], Train Loss: 0.5387, Val Loss: 0.4749
Epoch [20/500], Train Loss: 0.4629, Val Loss: 0.4715
Epoch [30/500], Train Loss: 0.4488, Val Loss: 0.4641
Epoch [40/500], Train Loss: 0.4405, Val Loss: 0.4675
Epoch [50/500], Train Loss: 0.4182, Val Loss: 0.4661
Early stopping at epoch 58 (best val loss: 0.4548)
Test Loss: 0.4028
Test MSE: 26.6866
Test R²: 0.5740
